# NSR Test: Faithful Dynamics vs. Simplified Adjacency Implementation

This notebook validates that `NSR_Dynamic.py` (faithful C-layer neural dynamics) and `NSR_Simplified.py` (adjacency-table simplification) learn the same compositional rules and produce the same query scores.

- **NSR_Dynamic**: explicit `X_C_agent(τ)` / `X_C_comp(τ)` evolution.
- **NSR_Simplified**: replaces the ERZ dynamics with equivalent adjacency lookups while keeping the same rule learning and scoring logic.

## Learning configuration hierarchy

Both `W_RR` learning and C-layer learning have a top-level dispatcher that selects one of several mutually-exclusive algorithms.

### 1. `W_RR` learning (`model.learn_WRR_hebbian(...)`)

`NSRConfig.W_RR_learning_algorithm` selects the algorithm:

```text
model.learn_WRR_hebbian(...)
│
├─ Top level: W_RR_learning_algorithm (mutually exclusive)
│   │
│   ├─ "online"       ──> _learn_WRR_online()
│   │                      standard per-pair Hebbian / Oja updates
│   │                      identical to NSR_Dynamic.py
│   │
│   └─ "cooccurrence" ──> _learn_WRR_cooccurrence()
│                          count relation co-occurrence on directed entity pairs
│                          much faster on large relation spaces, but NOT equivalent
│
├─ Sub-options: only effective when W_RR_learning_algorithm == "online"
│   │
│   ├─ delta_RR             Hebbian positive term
│   ├─ mu_RR                Oja decay term
│   ├─ W_RR_n_passes        number of passes
│   └─ W_RR_normalize       "sigmoid" / "max" / "none"
│
├─ Cross-cutting: shared by both algorithms
│   │
│   └─ W_RR_sigmoid_k       sigmoid steepness when normalize == "sigmoid"
│
└─ Legacy alias (backward compatible)
    │
    └─ W_RR_fast=True  -> W_RR_learning_algorithm = "cooccurrence"
```

### 2. C-layer learning (`model.learn_C_layer(...)`)

`NSRConfig.C_learning_algorithm` selects one of three mutually-exclusive algorithms:

```text
model.learn_C_layer(...)
│
├─ Top level: C_learning_algorithm (mutually exclusive)
│   │
│   ├─ "online"  ──> _learn_C_layer_online()
│   │                 standard per-path Hebbian / running_mean / EMA updates
│   │
│   ├─ "count"   ──> _learn_C_layer_count()
│   │                 one-pass explicit Python path counting
│   │                 mathematically equivalent to running_mean
│   │
│   └─ "matrix"  ──> _learn_C_layer_matrix()
│                     one-pass sparse-matrix path counting
│                     same result as "count" but faster on large/dense graphs
│
├─ Sub-options: only effective when C_learning_algorithm == "online"
│   │
│   ├─ C_fast_batching      batch updates by (h, t) endpoint
│   ├─ C_hebbian_mode       "running_mean" or "ema"
│   ├─ delta_C              EMA step size
│   ├─ C_n_passes           number of passes
│   └─ learn_cache_reverse  cache (h, t) -> forward relations
│
├─ Cross-cutting: shared by all three algorithms
│   │
│   ├─ C_dense_memory_limit_bytes   dense K³ tensor vs. Dict[(r_i,r_j), vec]
│   ├─ C_min_support                minimum support to keep a rule
│   ├─ C_rule_threshold             minimum confidence to keep a rule
│   └─ C_activity_threshold         threshold for active ERZ targets
│
└─ Legacy aliases (backward compatible)
    │
    ├─ C_fast_approx=True     -> C_learning_algorithm = "count"
    └─ C_fast_approx_v2=True  -> C_learning_algorithm = "matrix"
```

Recommended usage:

- `W_RR`: keep the default `"online"` for exact equivalence with `NSR_Dynamic.py`; use `"cooccurrence"` only when the relation space is very large and speed matters more than exact equivalence.
- C-layer small/medium graphs: keep the default `C_learning_algorithm="online"` or use `"count"` for a speedup.
- C-layer large relation spaces (e.g. FB15K-237): use `C_learning_algorithm="matrix"`.
- C-layer very large/dense graphs (e.g. YAGO3-10): use `C_learning_algorithm="matrix"`; consider forcing sparse storage with a small `C_dense_memory_limit_bytes`.

In [1]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "NSR_Dynamic.py").exists():
    NSR_DIR, REPO_ROOT = cwd, cwd.parent
elif (cwd / "NSR" / "NSR_Dynamic.py").exists():
    NSR_DIR, REPO_ROOT = cwd / "NSR", cwd
else:
    raise FileNotFoundError("Run this notebook from the repository root or the NSR directory.")
for source_dir in (NSR_DIR, REPO_ROOT / "NSR_CloseTo_Final"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))

import NSR_Dynamic as dynamic_module
import NSR_Simplified as simplified_module
from NSR_Dynamic import NSRConfig as DynConfig, NSRDynamicERZC as Dynamic
from NSR_Simplified import NSRConfig as SimConfig, NSRDynamicERZC as Simplified

## 3. Real dataset comparison: NATIONS

We compare train / query time and with-C MRR on NATIONS. Because the faithful dynamics is slower with all active rules, we also show `top_rules_fraction=0.05`.

In [2]:
from evaluate_dynamic_erz_c_optuna import load_dataset, build_filter_known, evaluate_filtered_tail

data = load_dataset("NATIONS", REPO_ROOT / "KINSHIP_1990")
print(f"NATIONS: N={data['N']}, M={data['M']}, train={len(data['train'])}, test={len(data['test'])}")

/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NATIONS: N=14, M=55, train=1592, test=201


In [3]:
from evaluate_dynamic_erz_c_optuna import make_config, train_model

params = {
    "T_thresh": 0.0226, "W_RR_sigmoid_k": 0.2153,
    "C_activity_threshold": 0.75, "C_rule_threshold": 0.3028,
    "C_min_support": 7, "agg_mode": "add", "alpha": 0.2919,
    "learn_cache_reverse": True,
}
seed = 42

t0 = time.perf_counter()
dyn_model, dyn_timing = train_model(dynamic_module, data, params, seed)
dyn_train = time.perf_counter() - t0

t0 = time.perf_counter()
sim_model, sim_timing = train_model(simplified_module, data, params, seed)
sim_train = time.perf_counter() - t0

print(f"Dynamic total train: {dyn_train:.2f}s, C train: {dyn_timing['c_train_sec']:.2f}s")
print(f"Simplified total train: {sim_train:.2f}s, C train: {sim_timing['c_train_sec']:.2f}s")
print(f"Train speedup: {dyn_train/sim_train:.1f}×")

Dynamic total train: 72.26s, C train: 72.13s
Simplified total train: 3.55s, C train: 3.54s
Train speedup: 20.4×


In [4]:
known = build_filter_known(data["train"], data["valid"], data["test"])
test = data["test"]

for top_frac in [1.0, 0.05]:
    t0 = time.perf_counter()
    dyn_res = evaluate_filtered_tail(dyn_model, test, known, use_c_layer=True, agg_mode="add", alpha=params["alpha"], seed=seed, top_rules_fraction=top_frac)
    dyn_q = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    sim_res = evaluate_filtered_tail(sim_model, test, known, use_c_layer=True, agg_mode="add", alpha=params["alpha"], seed=seed, top_rules_fraction=top_frac)
    sim_q = time.perf_counter() - t0
    
    print(f"\ntop_rules_fraction={top_frac}")
    print(f"  Dynamic MRR={dyn_res['MRR']:.4f}, time={dyn_q:.2f}s")
    print(f"  Simplified MRR={sim_res['MRR']:.4f}, time={sim_q:.2f}s, speedup={dyn_q/sim_q:.1f}×")

KeyboardInterrupt: 

## 4. Qualitative dynamic reasoning trace on NATIONS

This section selects a correctly predicted test triple whose gold tail is activated by relation equivalence through `W_RR`, with the C layer disabled. The stricter default also requires the gold tail to be the unfiltered Top-1 prediction and to have zero activation when `W_RR` is disabled. The table reports the Top-3 positive activations in the E, Z, and R layers at every synchronous ERZ time step.

In [5]:
from IPython.display import display
from NSR_Dynamic import gated_relu, sigma_gate

def trace_erz_states(model, head, relation, T=None, use_WRR=True):
    """Run the exact `_run_erz` recurrence and retain E/R/Z after each step."""
    head, relation = int(head), int(relation)
    steps = int(model.cfg.T_infer if T is None else T)
    I_E = np.zeros(model.N, dtype=float)
    I_R = np.zeros(model.K, dtype=float)
    I_E[head] = 1.0
    I_R[relation] = 1.0

    X_E = np.zeros(model.N, dtype=float)
    X_R = np.zeros(model.K, dtype=float)
    X_Z = np.zeros(model.NZ, dtype=float)
    zero_E = np.zeros(model.N, dtype=float)
    zero_R = np.zeros(model.K, dtype=float)
    trace = []
    cfg = model.cfg

    for tau in range(1, steps + 1):
        if cfg.hold_R_until_e_delay and tau <= max(1, int(cfg.i_e_delay)):
            inp_R = I_R.copy()
        elif tau == 1:
            inp_R = I_R.copy()
        else:
            inp_R = zero_R.copy()
        if tau >= int(cfg.inhibit_R_from):
            inp_R -= float(cfg.inhibit_value)
        inp_E = I_E if tau == int(cfg.i_e_delay) else zero_E

        zz_base = model.W_ZZ @ X_Z
        zz_drive = zz_base + float(cfg.b_bias) * (zz_base > 0.0).astype(float)
        ez_drive = model.W_EZ @ X_E
        rz_drive = model.W_ZR.T @ sigma_gate(X_R, float(cfg.T_thresh))
        X_Z_new = gated_relu(zz_drive + ez_drive + rz_drive, float(cfg.eta_Z))

        readout = X_Z.reshape(model.N, model.K).max(axis=1)
        X_E_new = np.clip(
            gated_relu(readout, float(cfg.eta_E)) + inp_E, 0.0, 1.0
        )
        rr_drive = model.W_RR @ X_R if use_WRR and cfg.use_rr_recurrence else zero_R
        X_R_new = np.clip(rr_drive + inp_R, 0.0, 1.0)

        X_E, X_R, X_Z = X_E_new, X_R_new, X_Z_new
        trace.append({"tau": tau, "E": X_E.copy(), "Z": X_Z.copy(), "R": X_R.copy()})

    # Guard against accidental drift from the implementation being visualized.
    expected = model.query(head, relation, T=steps, use_WRR=use_WRR, use_C_layer=False)
    observed = {i: float(X_E[i]) for i in range(model.N) if i != head and X_E[i] > 0.0}
    assert set(expected) == set(observed)
    assert all(np.isclose(expected[i], observed[i]) for i in expected)
    return trace

def _filtered_rank(scores, head, relation, tail, filter_known, num_entities):
    values = np.zeros(num_entities, dtype=float)
    for entity, score in scores.items():
        values[int(entity)] = float(score)
    for entity in filter_known.get((int(head), int(relation)), set()):
        if int(entity) != int(tail):
            values[int(entity)] = -np.inf
    return 1 + int(np.count_nonzero(values > values[int(tail)] + 1e-12))

def find_equivalence_only_examples(model, dataset, filter_known):
    """Correct no-C examples that require W_RR to activate the gold tail."""
    examples = []
    for test_index, (head, relation, tail) in enumerate(dataset["test"]):
        dynamic_scores = model.query(
            head, relation, use_WRR=True, use_C_layer=False
        )
        no_wrr_scores = model.query(
            head, relation, use_WRR=False, use_C_layer=False
        )
        gold_score = float(dynamic_scores.get(int(tail), 0.0))
        no_wrr_gold_score = float(no_wrr_scores.get(int(tail), 0.0))
        rank = _filtered_rank(
            dynamic_scores, head, relation, tail, filter_known, model.N
        )
        raw_rank = 1 + sum(
            float(score) > gold_score + 1e-12 for score in dynamic_scores.values()
        )
        if rank == 1 and gold_score > 0.0 and no_wrr_gold_score <= 0.0:
            examples.append({
                "test_index": int(test_index),
                "head_id": int(head),
                "relation_id": int(relation),
                "tail_id": int(tail),
                "head": dataset["entities"][int(head)],
                "relation": dataset["relations"][int(relation)],
                "tail": dataset["entities"][int(tail)],
                "gold_activation": gold_score,
                "without_WRR": no_wrr_gold_score,
                "filtered_rank": rank,
                "raw_rank": raw_rank,
            })
    return sorted(
        examples, key=lambda x: (x["raw_rank"], -x["gold_activation"], x["test_index"])
    )

In [6]:
def _top_positive(values, k=3):
    order = np.argsort(-np.asarray(values), kind="stable")
    return [(int(i), float(values[i])) for i in order[:k] if float(values[i]) > 0.0]

def _relation_label(model, dataset, relation_id):
    relation_id = int(relation_id)
    base = relation_id if relation_id < model.M else relation_id - model.M
    label = dataset["relations"][base]
    return label if relation_id < model.M else f"{label}^-1"

def _format_top(values, label_fn, k=3):
    items = _top_positive(values, k=k)
    return "[" + ", ".join(f"{{{label_fn(i)}: {value:.4f}}}" for i, value in items) + "]"

def make_dynamic_trace_table(model, dataset, example, k=3):
    head, relation, tail = example["head_id"], example["relation_id"], example["tail_id"]
    trace = trace_erz_states(model, head, relation, use_WRR=True)
    relation_name = lambda i: _relation_label(model, dataset, i)
    entity_name = lambda i: dataset["entities"][int(i)]
    z_name = lambda i: f"{relation_name(int(i) % model.K)}*{entity_name(int(i) // model.K)}"

    equivalent = [(i, value) for i, value in _top_positive(trace[1]["R"], k=model.K) if i != relation]
    equivalent_text = (
        f"{relation_name(equivalent[0][0])} ({equivalent[0][1]:.4f})" if equivalent else "no equivalent relation"
    )
    tail_z = trace[3]["Z"][tail * model.K:(tail + 1) * model.K]
    tail_z_top = _top_positive(tail_z, k=1)
    tail_z_text = (
        f"{relation_name(tail_z_top[0][0])}*{entity_name(tail)} ({tail_z_top[0][1]:.4f})"
        if tail_z_top else f"no active Z cell for {entity_name(tail)}"
    )

    meanings = [
        f"The query relation {relation_name(relation)} is clamped in R; the E input is deliberately delayed.",
        f"W_RR recurrently retrieves equivalent relations (strongest: {equivalent_text}) while {entity_name(head)} enters E.",
        f"Coincident E and R activity gates the relation-specific Z neurons of head entity {entity_name(head)}.",
        f"W_ZZ transfers activity to linked tail-side Z neurons; the gold path reaches {tail_z_text}.",
        f"The synchronous Z-to-E readout activates gold tail {entity_name(tail)} at {trace[-1]['E'][tail]:.4f}. C is disabled throughout.",
    ]

    rows = []
    for state, meaning in zip(trace, meanings):
        rows.append({
            "Time step": f"tau={state['tau']}",
            "E layer (Top-3)": _format_top(state["E"], entity_name, k),
            "Z layer (Top-3)": _format_top(state["Z"], z_name, k),
            "R layer (Top-3)": _format_top(state["R"], relation_name, k),
            "Interpretation": meaning,
        })
    return pd.DataFrame(rows)

trace_known = build_filter_known(data["train"], data["valid"], data["test"])
equivalence_examples = find_equivalence_only_examples(dyn_model, data, trace_known)
assert equivalence_examples, "No correctly predicted equivalence-only test example was found."
candidate_columns = [
    "test_index", "head", "relation", "tail", "gold_activation",
    "without_WRR", "filtered_rank", "raw_rank",
]
display(pd.DataFrame(equivalence_examples)[candidate_columns].head(10))

def show_equivalence_trace(test_index=None, k=3, export_markdown=False):
    """Display any candidate by its test_index; None selects the cleanest case."""
    if test_index is None:
        example = equivalence_examples[0]
    else:
        matches = [x for x in equivalence_examples if x["test_index"] == int(test_index)]
        if not matches:
            raise ValueError("test_index is not an equivalence-only Top-1 candidate listed above")
        example = matches[0]

    print(f"Selected test triple: ({example['head']}, {example['relation']}, {example['tail']})")
    print(
        f"Checks: C=OFF; filtered rank={example['filtered_rank']}; raw rank={example['raw_rank']}; "
        f"gold activation with W_RR={example['gold_activation']:.4f}, without W_RR={example['without_WRR']:.4f}."
    )
    table = make_dynamic_trace_table(dyn_model, data, example, k=k)
    display(table.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    if export_markdown:
        print("\nCopy-ready Markdown:\n")
        print(table.to_markdown(index=False))
    return table

# Interactive use: replace None with any test_index shown in the candidate table.
SELECTED_TEST_INDEX = None
dynamic_trace_table = show_equivalence_trace(SELECTED_TEST_INDEX, k=3)

,test_index,head,relation,tail,gold_activation,without_WRR,filtered_rank,raw_rank
0,47,cuba,timesincewar,usa,0.364382,0.0,1,1
1,112,israel,timesincewar,jordan,0.364382,0.0,1,1
2,136,netherlands,relintergovorgs,uk,0.340877,0.0,1,1
3,48,cuba,tourism3,ussr,0.175665,0.0,1,1
4,155,uk,duration,indonesia,0.157625,0.0,1,1
5,87,indonesia,expeldiplomats,cuba,0.122124,0.0,1,1
6,8,burma,dependent,uk,0.093502,0.0,1,1
7,132,netherlands,lostterritory,indonesia,0.065070,0.0,1,1
8,6,burma,blockpositionindex,usa,0.412750,0.0,1,2
9,161,uk,relngo,usa,0.390222,0.0,1,2


Selected test triple: (cuba, timesincewar, usa)
Checks: C=OFF; filtered rank=1; raw rank=1; gold activation with W_RR=0.3644, without W_RR=0.0000.


,Time step,E layer (Top-3),Z layer (Top-3),R layer (Top-3),Interpretation
0,tau=1,[],[],[{timesincewar: 1.0000}],The query relation timesincewar is clamped in R; the E input is deliberately delayed.
1,tau=2,[{cuba: 1.0000}],[],"[{timesincewar: 1.0000}, {timesincewar^-1: 0.3644}, {negativebehavior: 0.1885}]",W_RR recurrently retrieves equivalent relations (strongest: timesincewar^-1 (0.3644)) while cuba enters E.
2,tau=3,[],"[{timesincewar^-1*cuba: 0.3644}, {negativebehavior*cuba: 0.1885}, {negativebehavior^-1*cuba: 0.1650}]",[],Coincident E and R activity gates the relation-specific Z neurons of head entity cuba.
3,tau=4,[{cuba: 0.3644}],"[{timesincewar*usa: 0.3644}, {negativebehavior^-1*usa: 0.1885}, {negativebehavior*indonesia: 0.1650}]",[],W_ZZ transfers activity to linked tail-side Z neurons; the gold path reaches timesincewar*usa (0.3644).
4,tau=5,"[{usa: 0.3644}, {indonesia: 0.1650}, {brazil: 0.1479}]","[{relngo*china: 1.0000}, {intergovorgs3*cuba: 1.0000}, {ngoorgs3*cuba: 1.0000}]",[],The synchronous Z-to-E readout activates gold tail usa at 0.3644. C is disabled throughout.


In [ ]:
# Re-run with export_markdown=True to obtain a rebuttal-ready text table.
_ = show_equivalence_trace(SELECTED_TEST_INDEX, k=3, export_markdown=True)

## 5. Summary

- `NSR_Simplified.py` produces the same `W_comp_agent`, `C_support`, and `C_hit_count` as `NSR_Dynamic.py`.
- Query scores are identical on the synthetic KG.
- On NATIONS, training is significantly faster and the with-C MRR matches the faithful version.
- The dynamic qualitative trace isolates a correct test prediction that requires `W_RR` relation-equivalence dynamics while keeping the C layer disabled.
- The simplification is valid because the ERZ dynamics with W_RR disabled is equivalent to a direct-neighbor lookup, and the reverse consequent readout is equivalent to checking which forward relations connect `(h, t)`.